# CS156: Pipeline - First Draft

**Do I sound like AI, or have I always been a bad writer?**  

With the amount of AI we use on a day to day, I am fairly sure that a lot of the content I consume, whether it be the news, social media, or even the PCW that I grade, might be AI-assisted, if not completely AI-generated. This makes me wonder: as a symptom of consuming so much 'slop,' could my own writing also be showing traits of AI-generated text? 

This report answers this question by first training a model to classify assignments I wrote pre-Fall 2024, when AI tools have shown enough progress to make using it as an aid in coursework a viable strategy, from later submissions. Then, I use a (INSERT WHAT I ACTUALLY DO BASED ON PRELIMINARY MODEL RESULTS). 


## The Data

To create the dataset to be used in this pipeline, I retrieved past written assignments stored in my Minerva email's Google Drive using the notebook here ran in Colab ([GitHub](https://colab.research.google.com/drive/1f_YmZ3cR82UPcLH7uNJ9DUu-x7Gllc7c#scrollTo=e160UdxwwBmt)). Prior to the retreival, I manually labelled the documents that I wish to include. 

Importantly, since the focus of the models trained here is stylistic drift in *my* writing, I excluded the following assignments: 
- **Skill builders from CS111 and CS113:** The content is mostly mathematical equations and instructions, and they do not contain much original writing. In addition, they are hand-written documents, which adds more complexity to the ingestion process. 
- **Group assignments:** These contain writing by other students, thus were excluded to limit the training data to my own writing. 
- **Heavily templated assignments:** All assignments formatted as workbooks (both Forum codebooks and worksheets with instructions included) were excluded to prevent 'dilution' of my writing due to the instruction text included in the document. This resulted in the exclusion of several FA50 and FA51 (previously CS50 and CS51) assignments, Deep Dives from CS111 and CS113, problems sets from CS114, and some SS110 and SS152 assginments that are formatted as cover sheets. 

For the same reasons, I included any non-essay documents that reflect my own prose, such as pre-written scripts for any videos I submitted as assignments, or reflection documents that supplement technical interviews. 

Based on these exclusion criteria, 57 assignment documents qualified for inclusion in the dataset. However, 4 of these assignments were created outside of Google Drive (e.g., PDFs exported from OverLeaf), and I excluded them from the final dataset. Extracting text from PDFs would require different data processing compared to Google Drive documents, and given the small number of such files, I chose to maintain consistency by excluding them entirely.

## Loading the Data

Using (HOW DO I SCRAPE GDRIVE LOOK THIS UP), I loaded the text data from my past assignments in to a pandas dataframe and tokenized (UPDATE AS I ADD CODE). 

In [ ]:
# Install necessary packages
!pip install pandas numpy scikit-learn matplotlib seaborn scipy textstat nltk

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 33.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 24.1 MB/s eta 0:00:0000:0100:01
  Using cached scikit_learn-1.8.0-cp311-cp311-macosx_12_0_arm64.whl (8.1 MB)
  Using cached matplotlib-3.10.8-cp311-cp311-macosx_11_0_arm64.whl (8.1 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 23.5 MB/s eta 0:00:00
  Using cached scipy-1.17.0-cp311-cp311-macosx_14_0_arm64.whl (20.1 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 23.5 MB/s eta 0:00:00a 0:00:01
  Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
  Using cached contourpy-1.3.3-cp311-cp311-macosx_11_0_arm64.whl (270 kB)
  Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
  Using cached fonttools-4.61.1-cp311-cp311-macosx_10_9_universal2.whl (2.9 MB)
  Using cached kiwisolve

In [ ]:
# Imports

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import re

# Sklearn imports
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.decomposition import PCA

# Stylometric analysis
import textstat
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from collections import Counter

In [49]:
# Loading dataset

df = pd.read_csv("gdrive_scraped_data.csv", encoding='latin-1')
df.head()


file_id  \
0  1z4dvX8KX5spMNnUNkWoboNuA6doneGrNYJJWlvB2Wno   
1  19du57lm4DB4yeZuFAlyTAdgp3u-ug8MV4eLIbQNwZk8   
2  1yNA1ZfpoNLkbVhCHcuCIGBHFT1eCHsjKcmN4_icO7Yw   
3  16AVkozInKE_ol_qFQMXxQOy3OMK5kHQjDvgRUQ1cUSk   
4  1gBiAfz7LWm-IePjnEvHuelT03Uocw6l9ywwbvunsxX4   

                                        name              created_time  \
0        Reflection on Track Options [final]  2026-01-30T19:51:16.879Z   
1                                LBA [final]  2026-02-07T12:24:25.590Z   
2  Elevation Reflection & Engagement [final]  2026-01-07T08:37:17.604Z   
3            Ethnographic Assignment [final]  2026-02-10T13:05:32.687Z   
4                   Final assignment [final]  2025-12-04T22:18:34.515Z   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        

## Pre-processing and EDA

### Data Cleaning to Prevent Leakage

To ensure the classifier learns stylistic patterns rather than metadata, I create multiple versions of the text with different levels of cleaning:

**Text columns created:**
- `text_raw`: Original text with all content (baseline)
- `no_cover`: Cover sheet removed only (strips course code, instructor name, submission date)
- `text_clean`: Fully cleaned - removes cover sheet AND all formatting elements (AI statements, references, appendices, footnotes including HC tags)

**Metadata extracted:**
- `course_code`: Extracted from cover sheet for one-hot encoding

**Rationale for cleaning:
- **Cover sheet removal**: Eliminates temporal markers (dates, instructors) and course identifiers that could leak time periods
- **Formatting removal**: Strips references, appendices, footnotes (including HC applications), and AI statements. These elements represent assignment structure rather than core writing style, and their presence/absence could leak information about course requirements or time periods

This approach allows comparison of model performance on raw text vs. main body prose to determine whether the classifier learns from content/metadata or actual writing style.

In [ ]:
# Preprocessing functions

VALID_COURSES = ['CP192', 'CS146', 'GL96', 'CS156', 'SS111',
                 'CP191', 'CS166', 'GL95', 'SS152', 'SS156', 
                 'CS113', 'CS114', 'GL94', 'SS110', 
                 'CS110', 'CS111', 'GL93', 'SS112', 
                 'CX51', 'EA51', 'FA51', 'GL92', 'MC51', 
                 'CX50', 'EA50', 'FA50', 'GL91', 'MC50', 'IL199']

def extract_course_code(text):
    """
    Extract course code from cover sheet (second line under title).
    Returns tuple: (course_code, needs_review_flag)
    """
    # Look for pattern: 2 letters + 2-3 digits
    # Search in first 500 characters (cover sheet area)
    first_section = text[:500]
    
    # Pattern: course prefix (2 letters) + course number (2-3 digits)
    pattern = r'\b([A-Z]{2})[\s-]?(\d{2,3})\b'
    matches = re.findall(pattern, first_section)
    
    if matches:
        # Reconstruct course code
        prefix, number = matches[0]
        course_code = f"{prefix}{number}"
        
        # Check if it's in our valid list
        needs_review = course_code not in VALID_COURSES
        return course_code, needs_review
    
    return None, True  # Flag for manual review if not found

def remove_cover_sheet(text):
    """
    Remove cover sheet (up to first major page break).
    Page breaks are multiple \\r\\n with optional horizontal rule.
    """
    # Look for pattern: multiple line breaks (6+) with optional horizontal line
    page_break_pattern = r'(\r?\n){6,}(_+)?(\r?\n)+'
    
    match = re.search(page_break_pattern, text)
    if match:
        # Return everything after the first page break
        return text[match.end():]
    
    # Fallback: No page break found
    # Look for course code and remove everything before it
    course_code_pattern = r'\b([A-Z]{2})[\s-]?(\d{2,3})\b'
    match = re.search(course_code_pattern, text)
    
    if match:
        # Find the line containing the course code
        course_code_pos = match.start()
        # Find the next newline after the course code (end of that line)
        next_newline = text.find('\n', course_code_pos)
        if next_newline != -1:
            return text[next_newline+1:]  # Return everything after the course code line
    
    return text  # Return as-is if nothing found

def remove_hc_tags(text):
    """
    Remove HC tags and their applications.
    Pattern: #tag followed by : and explanation text until double newline.
    """
    # Pattern: # followed by alphanumeric/hyphens, then : and everything until double newline
    # This captures: #ss112-institutionalism: explanation text\n\n
    pattern = r'\#[A-Za-z0-9\-]+:[^\n]*(?:\n(?!\n)[^\n]*)*'
    return re.sub(pattern, '', text)

def extract_body_only(text):
    """
    Extract main body text by stopping at 'Word Count:' or 'AI Statement:'.
    This removes AI statements, references, and appendices.
    """
    # Look for "Word Count:" or "AI Statement:" (case-insensitive)
    pattern = r'(?i)(word count|ai statement|ai use)'
    
    match = re.search(pattern, text)
    if match:
        return text[:match.start()].strip()
    
    return text  # Return full text if no marker found

def preprocess_dataframe(df):
    """
    Apply all preprocessing steps to create new text columns.
    
    Creates columns:
    - text_raw (already exists) - Baseline with everything
    - no_cover: Cover sheet removed only
    - text_clean: Fully cleaned (no cover + formatting elements such as HC footnotes, references, appendices removed)
    - course_code: extracted course code
    - has_ai_statement: boolean flag
    """
    df_processed = df.copy()
    
    # Extract course codes
    course_info = df_processed['text_raw'].apply(extract_course_code)
    df_processed['course_code'] = course_info.apply(lambda x: x[0])
    """
    df_processed = df.copy()
    
    # Extract course codes
    course_info = df_processed['text_raw'].apply(extract_course_code)
    df_processed['course_code'] = course_info.apply(lambda x: x[0])
    
    # Create text columns
    df_processed['no_cover'] = df_processed['text_raw'].apply(remove_cover_sheet)
    
    # Apply all cleaning: remove body end (Word Count onwards) + HC tags/applications
    df_processed['text_clean'] = df_processed['no_cover'].apply(extract_body_only).apply(remove_hc_tags)
    
    # Metadata flags
    return df_processed

def create_course_onehot(df, column='course_code'):
    """
    Create one-hot encoding for course codes.
    Returns dataframe with course_XXX columns.
    Create one-hot encoding for course codes.
    return pd.get_dummies(df[column], prefix='course')

# Apply preprocessing
df_processed = preprocess_dataframe(df)
# Apply preprocessing
# Display summary
print("Preprocessing complete.")
print(f"\nTotal documents: {len(df_processed)}")

# Display head with truncated text columns for readability
display_df = df_processed.head().copy()
text_cols = ['text_raw', 'no_cover', 'text_clean']
for col in text_cols:

Preprocessing complete.

Total documents: 53


,file_id,name,created_time,text_raw,course_code,no_cover,text_clean,has_ai_statement
0,1z4dvX8KX5spMNnUNkWoboNuA6doneGrNYJJWlvB2Wno,Reflection on Track Options [final],2026-01-30T19:51:16.879Z,ï»¿Reflection on Track Options\r\n\r\n\r\n\r\n\r\nMinerva University\r\nCP192: Capstone Seminar\r\nProf. L. Odera\r...,CP192,"Reflection on Track Options\r\nProcess Documentation\r\nTo ideate what kind of track to propose, I answe...","Reflection on Track Options\r\nProcess Documentation\r\nTo ideate what kind of track to propose, I answe...",True
1,19du57lm4DB4yeZuFAlyTAdgp3u-ug8MV4eLIbQNwZk8,LBA [final],2026-02-07T12:24:25.590Z,ï»¿LBA: Analyzing the Bangle Market of Charminar\r\n\r\n\r\n\r\n\r\nMinerva University\r\nSS111: Modern Economic...,SS111,"LBA: Analyzing the Bangle Market of Charminar\r\n Located in the âOld Cityâ of Hyderabad, I...","LBA: Analyzing the Bangle Market of Charminar\r\n Located in the âOld Cityâ of Hyderabad, I...",True
2,1yNA1ZfpoNLkbVhCHcuCIGBHFT1eCHsjKcmN4_icO7Yw,Elevation Reflection & Engagement [final],2026-01-07T08:37:17.604Z,ï»¿Elevation Reflection & Engagement \r\nGL96\r\nSuisei Nakagawa\r\n\r\n\r\nOne way I plan to engage meaningfu...,GL96,Suisei Nakagawa\r\n\r\n\r\nOne way I plan to engage meaningfully with Hyderabad is through its local climb...,Suisei Nakagawa\r\n\r\n\r\nOne way I plan to engage meaningfully with Hyderabad is through its local climb...,False
3,16AVkozInKE_ol_qFQMXxQOy3OMK5kHQjDvgRUQ1cUSk,Ethnographic Assignment [final],2026-02-10T13:05:32.687Z,"ï»¿Ethnographic Assignment\r\n\r\n\r\n\r\n\r\nMinerva University\r\nGL96: Global Learning\r\nJanuary 9, 2025\r\n\r\n\r\n...",GL96,"Ethnographic Assignment\r\nIntroduction\r\nFor this assignment, I spoke to Arjun Rao, a regular at my cl...","Ethnographic Assignment\r\nIntroduction\r\nFor this assignment, I spoke to Arjun Rao, a regular at my cl...",True
4,1gBiAfz7LWm-IePjnEvHuelT03Uocw6l9ywwbvunsxX4,Final assignment [final],2025-12-04T22:18:34.515Z,ï»¿Tab 1\r\n\r\n\r\n\r\n\r\n\r\n\r\nComparative Analysis of Political Systems: Furusao Nozei in Japan\r\n\r\n\r\n\r\n\r\nMin...,SS156,Comparative Analysis of Political Systems: Furusao Nozei in Japan\r\n\r\n\r\n\r\n\r\nMinerva University\r\nSS156...,Comparative Analysis of Political Systems: Furusao Nozei in Japan\r\n\r\n\r\n\r\n\r\nMinerva University\r\nSS156...,True


In [57]:
# Display last 500 chars of cleaned text for FA50 assignments
course = 'MC51'
filtered = df_processed[df_processed['course_code'] == course]

print(f"{course} assignments: {len(filtered)}\n")
print("="*80)

for idx, row in filtered.iterrows():
    print(f"\nDocument: {row['name']}")
    print("-"*80)
    print(row['text_clean'][-100:])
    print("="*80)

MC51 assignments: 5


Document: Writing Journal 1 [final]
--------------------------------------------------------------------------------
two-dimensional. The solid printed colors give the cover a matte texture.[1]
________________
[1] 

Document: Writing Journal 2 [final]
--------------------------------------------------------------------------------
is journal entry and TA office hours as an opportunity to practice the prospective approach.


* 

Document: Script [final]
--------------------------------------------------------------------------------
PMC. 
The Most Important Fast Fashion Statistics [2024]. 
Minimum Wage | U.S. Department of Labor.

Document: LBA: Multimodal Persuasion [final]
--------------------------------------------------------------------------------
ssiveness, organizational transparency centered on sustainable values, and low costs.
HC Appendix


Document: Interpretation with #medium + #context [final]
--------------------------------------------------------

In [ ]:
# Find all instances of "AI" in cleaned text to verify AI statements are removed

# Prevent pandas truncation
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

doc_count = 0
total_matches = 0

print("Documents containing 'AI' in cleaned text:\n")
print("="*80)

for idx, row in df_processed.iterrows():
    # Find all matches of "AI" with 100 chars context after
    text = row['text_clean']
    matches = [(m.start(), text[m.start():m.start()+102]) for m in re.finditer(r'\bAI\b', text)]
    
    if matches:
        doc_count += 1
        total_matches += len(matches)
        
        print(f"\n📄 {row['name']}")
        print(f"   Course: {row['course_code']}")
        print(f"   Found {len(matches)} instance(s):")
        print("-"*80)
        
        for i, (pos, context) in enumerate(matches, 1):
            print(f"   [{i}] Position {pos}: {context}")
            print()

print("="*80)
print(f"\n📊 SUMMARY:")
print(f"   Documents with 'AI': {doc_count}")
print(f"   Total 'AI' instances: {total_matches}")
print("="*80)

Documents containing 'AI' in cleaned text:


📄 Process documentation [final]
   Course: CP191
   Found 1 instance(s):
--------------------------------------------------------------------------------
   [1] Position 931: AI
I utilized GitHub Copilot in multiple parts of my process. I chose Copilot as my primary tool beca


📄 Elevation Reflection & Engagement [final]
   Course: GL95
   Found 1 instance(s):
--------------------------------------------------------------------------------
   [1] Position 1594: AI, was the dominant industry and companies ranged from niche pre-seed startup to multi-billion-dollar


📄 Reflection [final]
   Course: CS111
   Found 2 instance(s):
--------------------------------------------------------------------------------
   [1] Position 545: AI tools, specifically ChatGPT, to help me with SageMath syntax. I typically used prompts such as âh

   [2] Position 1259: AI has overall made my learning more efficient but will be even more helpful if I could put mo

In [ ]:
# Find all instances of "#" in cleaned text to verify HC tags are removed

# Prevent pandas truncation
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

doc_count = 0
total_matches = 0

print("Documents containing '#' in cleaned text:\n")
print("="*80)

for idx, row in df_processed.iterrows():
    # Find all matches of "#" with 100 chars context after
    text = row['text_clean']
    matches = [(m.start(), text[m.start():m.start()+102]) for m in re.finditer(r'#([A-Za-z0-9]+)', text)]
    
    if matches:
        doc_count += 1
        total_matches += len(matches)
        
        print(f"\n📄 {row['name']}")
        print(f"   Course: {row['course_code']}")
        print(f"   Found {len(matches)} instance(s):")
        print("-"*80)
        
        for i, (pos, context) in enumerate(matches, 1):
            print(f"   [{i}] Position {pos}: {context}")
            print()

print("="*80)
print(f"\n📊 SUMMARY:")
print(f"   Documents with '#': {doc_count}")
print(f"   Total '#' instances: {total_matches}")
print("="*80)

Documents containing '#' in cleaned text:


📊 SUMMARY:
   Documents with '#': 0
   Total '#' instances: 0


In [55]:
# Display first 100 chars of cleaned text
for idx, row in df_processed.iterrows():
    print(f"\n📄 {row['name']}")
    print("-"*80)
    print(row['text_clean'][:200])
    print("="*80)


📄 Reflection on Track Options [final]
--------------------------------------------------------------------------------
Reflection on Track Options
Process Documentation
To ideate what kind of track to propose, I answered the provided questions below. 
Of the tracks that would fulfill the requirements of your major(

📄 LBA [final]
--------------------------------------------------------------------------------
LBA: Analyzing the Bangle Market of Charminar
        Located in the âOld Cityâ of Hyderabad, India, the Charminar is known for its bustling outdoor markets. This paper focuses on the market for 

📄 Elevation Reflection & Engagement [final]
--------------------------------------------------------------------------------
Suisei Nakagawa


One way I plan to engage meaningfully with Hyderabad is through its local climbing community. I began bouldering in Spring 2025 and have since climbed regularly in gyms in San Fra

📄 Ethnographic Assignment [final]
---------------------------

## Analysis

## Model Selection 

## Training 

In [ ]:
#@title also session 5 PCW (CHANGE LATER)

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import metrics, naive_bayes

labels, texts = df['v1'], df['v2']
vectorizer = text.TfidfVectorizer()
vec_texts = vectorizer.fit_transform(texts)


# We train a model
model = naive_bayes.MultinomialNB()
model.fit(vec_texts, labels)
pred_labels = model.predict(vec_texts)

confusion_mat = metrics.confusion_matrix(labels, pred_labels)
sns.heatmap(confusion_mat, square=True, annot=True, fmt='d', cbar=False)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

# some checks to see how the model is doing
print(model.classes_)
print(df['v1'].value_counts())
print(vec_texts.shape) 




## Predictions

## Discussion 

## Summary 

## References 

- Session 5 PCW: https://forum.minerva.edu/app/courses/3804/sections/13026/classes/99436 